# Notebook 13 — Consumindo APIs

Uma **API** é uma forma de programas conversarem pela internet. Aqui aprendo a ser
o **cliente**: enviar requisições e usar as respostas. No próximo notebook viro o
**servidor**.

## Conteúdo
- o que é uma API e HTTP;
- métodos (GET, POST, PUT, DELETE);
- status codes;
- a biblioteca `requests`;
- lendo respostas em JSON.

> Algumas células fazem chamadas reais à internet. Para o notebook rodar mesmo
> offline (ex.: no CI), elas estão protegidas com `try/except`.

## 1. Conceitos de HTTP
Quando acesso uma API, envio uma **requisição** e recebo uma **resposta**.

**Métodos HTTP** (o que quero fazer):
- `GET` — buscar dados;
- `POST` — criar;
- `PUT`/`PATCH` — atualizar;
- `DELETE` — apagar.

**Status codes** (como foi):
- `2xx` sucesso (200 OK, 201 Created);
- `4xx` erro do cliente (404 Não encontrado, 400 Requisição inválida);
- `5xx` erro do servidor.

## 2. A resposta vem em JSON
Antes mesmo de ir à internet: APIs quase sempre respondem em JSON, que eu já sei
converter para dicionário (notebook 09).

In [ ]:
import json

# exemplo de corpo de resposta que uma API devolveria
resposta_json = '''
{
  "id": 1,
  "title": "Estudar Python",
  "completed": false
}
'''

tarefa = json.loads(resposta_json)
print("título:", tarefa["title"])
print("concluída?", tarefa["completed"])

## 3. Fazendo um GET com requests
A biblioteca `requests` é o jeito mais comum de consumir APIs em Python.
Uso a API pública de teste **JSONPlaceholder**.

In [ ]:
import requests

try:
    resposta = requests.get("https://jsonplaceholder.typicode.com/todos/1", timeout=10)
    print("status:", resposta.status_code)
    dados = resposta.json()          # converte o JSON em dicionário
    print("dados:", dados)
except requests.RequestException as erro:
    print("Sem internet agora — em um ambiente conectado isto traria a tarefa 1.")
    print("Detalhe:", erro)

## 4. Tratando o status code
Sempre verifique se deu certo antes de usar os dados.

In [ ]:
import requests

try:
    r = requests.get("https://jsonplaceholder.typicode.com/todos/1", timeout=10)
    if r.status_code == 200:
        print("OK! título:", r.json()["title"])
    elif r.status_code == 404:
        print("recurso não encontrado")
    else:
        print("algo deu errado:", r.status_code)
except requests.RequestException:
    print("(offline) — fluxo de verificação de status demonstrado acima")

## 5. Enviando dados com POST
Para criar um recurso, mando um corpo (geralmente JSON) no método POST.

In [ ]:
import requests

nova_tarefa = {"title": "Aprender APIs", "completed": False, "userId": 1}

try:
    r = requests.post(
        "https://jsonplaceholder.typicode.com/todos",
        json=nova_tarefa,            # 'json=' já serializa e ajusta o cabeçalho
        timeout=10,
    )
    print("status:", r.status_code)  # 201 = criado
    print("resposta:", r.json())
except requests.RequestException:
    print("(offline) — em um ambiente conectado, criaria a tarefa e retornaria 201")

## 6. Query params e cabeçalhos
- **params**: filtros que vão na URL (`?userId=1`);
- **headers**: metadados, como autenticação (`Authorization`).

In [ ]:
import requests

try:
    r = requests.get(
        "https://jsonplaceholder.typicode.com/todos",
        params={"userId": 1},
        headers={"Accept": "application/json"},
        timeout=10,
    )
    tarefas = r.json()
    print(f"o usuário 1 tem {len(tarefas)} tarefas")
    print("URL final:", r.url)
except requests.RequestException:
    print("(offline) — a URL final seria .../todos?userId=1")

## Conclusão
Sei consumir APIs: fazer GET/POST, enviar parâmetros e ler JSON. Agora, no último
notebook, vou para o outro lado e **construir** minha própria API com FastAPI.